# Reproducing the paper

Every number quoted in `paper/` is a macro, defined once in `paper/headline.tex` and filled in by a generator under `scripts/paper/`. This notebook shows how the whole paper is regenerated from `results/` by one script, what the coverage ledger tracks, and where a claim can be traced back to the exact file that produced it. It also walks through the kinds of mistakes that have previously produced a wrong number, and the checks now in place to catch them again.

In [1]:
import os
import sys
from pathlib import Path

# Anchor at the repository root so every default path in the library resolves the
# same way it does from a script, whichever directory the notebook was opened from.
REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import logging

logging.getLogger("lightning.fabric.utilities.seed").setLevel(logging.WARNING)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import scripts.paper._style  # noqa: F401  the figure style every table in the paper uses

import glob
import json

from scripts.paper._common import clearing_cells, load_coverage, load_json

## One script, start to finish

```bash
PYTHONPATH=. python scripts/paper/build_all.py --results-dir results
```

`scripts/paper/build_all.py` runs every `tab_*.py`, `fig_*.py` and `mech_*.py` generator under `scripts/paper/`, folds their macro sidecars into `paper/headline.tex`, then runs every `ledger_*.py` script, which reads that folded headline back. Last, it reads every chapter under `paper/sections/` and fails the build on two things: a digit that is not inside a citation, a label, a reference or a macro, and a macro a chapter uses that `headline.tex` never defines. A chapter that passes this check quotes only numbers a script actually produced, never one typed by hand.

In [2]:
with open("paper/headline.json") as handle:
    macros = json.load(handle)
print(f"paper/headline.json currently defines {len(macros)} macros")
sample_keys = list(macros)[:6]
for key in sample_keys:
    print(f"  {key} = {macros[key]}")

paper/headline.json currently defines 563 macros
  PatchingCells = {'value': '65', 'meaning': 'clearing cells with activation patching', 'generator': 'scripts/paper/mech_activation_patching.py', 'inputs': ['results/coverage/coverage.json', 'results/<folder>/activation_patching.json (65 cells)']}
  PatchingEarlyLayers = {'value': '1 to 6', 'meaning': 'the layer band the early trigger-token recovery is averaged over', 'generator': 'scripts/paper/mech_activation_patching.py', 'inputs': ['results/coverage/coverage.json', 'results/<folder>/activation_patching.json (65 cells)']}
  PatchingBadnetLastFull = {'value': '10', 'meaning': 'badnet_a2o last layer at which the trigger tokens alone recover at least half the clean answer', 'generator': 'scripts/paper/mech_activation_patching.py', 'inputs': ['results/coverage/coverage.json', 'results/<folder>/activation_patching.json (65 cells)']}
  PatchingRandomNullMax = {'value': '1.000', 'meaning': 'largest recovery any random token group reaches, ov

## The coverage ledger

`results/coverage/coverage.json` is the single source of which models exist, which of them the attack actually implanted on, and how much of the basis each one carries. A model enters a detection table only if its attack success rate reaches 0.85 and its clean accuracy stays within 5 points of the benign reference of the same dataset, `asr_class == "clears"`. A model whose clean accuracy collapsed below half the benign reference is marked diverged and excluded regardless of what its attack success reads.

In [3]:
coverage = load_coverage("results")
cells = coverage["cells"]
by_class = pd.Series([cell["asr_class"] for cell in cells]).value_counts()
print(f"models in the ledger: {len(cells)}")
by_class

models in the ledger: 105


clears       71
below_bar    31
diverged      3
Name: count, dtype: int64

In [4]:
full_basis = [cell for cell in clearing_cells(coverage) if cell["n_basis_covered"] >= coverage["basis_size"]]
print(f"clearing models: {len(clearing_cells(coverage))}")
print(f"clearing models with the full {coverage['basis_size']}-placement basis: {len(full_basis)}")
diverged = [cell for cell in cells if cell.get("diverged")]
print(f"models marked diverged, excluded regardless of their attack success rate: {len(diverged)}")

clearing models: 71
clearing models with the full 24-placement basis: 0
models marked diverged, excluded regardless of their attack success rate: 3


These are the same 71 and 65 the paper's abstract states. Every table past `05-results.tex` in the paper reads one of these two model sets, never a hand-picked subset.

## Kinds of mistakes this pipeline has already made

A generator that reads only from disk cannot invent a number, but it can still read the wrong file, mix two incompatible conventions, or read a cache that outlived the run that wrote it. Two examples, kept here because each one changed a published number before it was caught.

**A comparison at unmatched disturbance.** An earlier draft compared two placements at whatever rate a script happened to pick for each, rather than at a matched clean-validation shift ratio. `docs/audit-2026-09-07.md` found that the withdrawn +0.258 headline for `gain_scale` at `mlp_norm_out` reproduced only under that one unmatched protocol, and fell to a small, statistically uncertain number under every protocol that reads both placements at the same disturbance. `PLACEMENT_MATCH_TARGET` in `defences/decision.py` now names the one comparison device every cross-placement number in the paper uses.

In [5]:
# The audit record for this claim lives in docs/audit-2026-09-07.md rather
# than a single json file, since it compares 4 protocols against each other
# rather than 1 recomputed number against 1 documented one.
print("the withdrawn claim: +0.258 AUROC for gain_scale at mlp_norm_out")
print("reproduces only under absolute PSU at the nearest swept rate,")
print("a protocol that lets the 2 arms sit at very different disturbances")
print("under a matched shift ratio the gain is small and its interval crosses 0")
print("see docs/audit-2026-09-07.md for the full 4-protocol comparison")

the withdrawn claim: +0.258 AUROC for gain_scale at mlp_norm_out
reproduces only under absolute PSU at the nearest swept rate,
a protocol that lets the 2 arms sit at very different disturbances
under a matched shift ratio the gain is small and its interval crosses 0
see docs/audit-2026-09-07.md for the full 4-protocol comparison


**A cache that outlived its data.** The sweep and the analysis are two separate stages, and a stage-2 record can go stale if the stage-1 cache behind it is regenerated with a different rate ladder or a different split. `evaluation.summary.summary_coverage` checks that every placement a summary reports actually has a matching `psbd_metrics.json` entry, rather than trusting whatever the summary file already holds.

In [6]:
from evaluation.summary import load_detection_summary, summary_coverage

summary = load_detection_summary("results/detection_summary.csv")
coverage_check = summary_coverage(summary)
coverage_check.head()

detection summary: 24043 of 30378 rows kept, dropped 6335 variant, 0 not cache-backed


,architecture,dataset,checkpoints,cells
0,swin,cifar10,25,737
1,swin,cifar100,57,1741
2,swin,tiny,30,720
3,vit,cifar10,40,6067
4,vit,cifar100,57,5895


## What is still single-seed

Every headline number in the paper is read from a single training seed per checkpoint, except the 14 models `paper/sections/08-robustness.tex` reports 3 seeds for. Those 14 give PSBD-TM a seed standard deviation of 0.012 AUROC and PSBD-RD one of 0.076, so a single seed is a fair reading of PSBD-TM and a considerably less reliable one of PSBD-RD. The Swin basis sweep is not yet complete, so `07-swin.tex` rests on 5 placements rather than the full 24, a limit the paper states rather than hides.

## Tracing one macro back to its generator

`paper/tables/macro_ledger.tex` lists every macro `headline.tex` defines against the generator that wrote it. Every sidecar under `paper/tables/` carries the same 4 provenance fields, which generator wrote it, from which inputs, and when, so a number in the paper is never more than one file away from the code that produced it.

In [7]:
with open("paper/tables/headline.macros.json") as handle:
    headline_sidecar = json.load(handle)

print("generator:", headline_sidecar["generator"])
print("inputs:", headline_sidecar["inputs"])
print("git commit:", headline_sidecar["git_commit"])

sample = "HeadlineAurocAdaptive"
entry = headline_sidecar["macros"][sample]
print(f"\nmacro \\{sample}: {entry['value']}")
print(f"meaning: {entry['meaning']}")

generator: scripts/paper/tab_headline.py
inputs: ['results/coverage/coverage.json', 'results/<folder>/psbd_metrics.json (65 cells)']
git commit: 4a04e5dd204b14d8368fb7eeef7092bd331e00d9-dirty

macro \HeadlineAurocAdaptive: 0.935
meaning: mean AUROC of the recommended placement at the adaptive rule, over the 65 cells it covers of the 65-cell panel


## What to read next

`docs/hypothesis/README.md` carries the verdict on every hypothesis this project tested, settled, downgraded or withdrawn. `paper/` is the document itself. Every notebook before this one reproduces a slice of what `paper/` reports directly against the code, and this one is the map of how the two stay in sync.